<cell_type>markdown</cell_type># Step 4: After 평가 (Fine-tuned 모델 성능 비교)

## 학습 목표
이 노트북을 완료하면 다음을 이해할 수 있습니다:
- 각 **Fine-tuning 기법별 성능 차이** 분석
- **SageMaker Model Registry** 사용법
- 모델 버전 관리와 **승인 워크플로우**

## 평가 프로세스

```
S3에 저장된 모델들
      ↓
┌─────────────────┐
│  모델 다운로드   │  model.tar.gz → model.pth
└────────┬────────┘
         ↓
┌─────────────────┐
│  모델 로드      │  PyTorch 모델에 가중치 로드
└────────┬────────┘
         ↓
┌─────────────────┐
│  테스트 평가    │  동일한 테스트 데이터로 평가
└────────┬────────┘
         ↓
┌─────────────────┐
│  결과 비교      │  Accuracy, Precision, Recall, F1
└────────┬────────┘
         ↓
┌─────────────────┐
│  Model Registry │  85%+ 모델만 등록
└─────────────────┘
```

## 예상 결과
- **Full Fine-tuning**: ~90%+ (가장 높은 성능)
- **Layer Freezing**: ~80-85% (빠르고 안정적)
- **LoRA**: ~85-90% (효율적)

In [ ]:
import json
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import boto3
import tarfile
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '4_after_evaluation'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# 설정 로드
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Config loaded: {config_path}")

In [ ]:
# ============================================
# 📥 S3에서 Fine-tuned 모델들 다운로드
# ============================================
import sagemaker

s3 = boto3.client('s3')

# 학습된 모델 정보 로드
training_results = config.get('training_results', {})

if not training_results:
    # 단일 모델만 있는 경우 (이전 버전 호환)
    training_results = {
        'full': {
            'model_data': config['model_data'],
            'training_job_name': config.get('training_job_name', 'unknown')
        }
    }

print("학습된 모델 목록:")
for method, info in training_results.items():
    print(f"  🔹 {method.upper()}: {info['model_data']}")

# 각 모델 다운로드
import os
os.makedirs('./models', exist_ok=True)

for method, info in training_results.items():
    model_data = info['model_data']
    local_tar = f'./models/{method}_model.tar.gz'
    local_dir = f'./models/{method}'
    
    print(f"\n{method.upper()} 모델 다운로드 중...")
    !aws s3 cp {model_data} {local_tar}
    
    os.makedirs(local_dir, exist_ok=True)
    !tar -xzf {local_tar} -C {local_dir}
    print(f"✅ {method.upper()} 모델 다운로드 완료")

print("\n모든 모델 다운로드 완료!")

In [ ]:
# ============================================
# 🔧 Fine-tuned 모델들 로드
# ============================================

class DeepfakeDetector(nn.Module):
    """EfficientNet 기반 딥페이크 탐지 모델"""
    def __init__(self, model_name='efficientnet_b0', num_classes=2, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    def forward(self, x):
        return self.backbone(x)

# 각 기법별 모델 로드
models = {}

for method in training_results.keys():
    model_path = f'./models/{method}/model.pth'
    
    # best_model.pth가 있으면 사용
    best_model_path = f'./models/{method}/best_model.pth'
    if os.path.exists(best_model_path):
        model_path = best_model_path
    
    print(f"{method.upper()} 모델 로드 중: {model_path}")
    
    model = DeepfakeDetector(pretrained=False)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    models[method] = model
    print(f"✅ {method.upper()} 모델 로드 완료")

print(f"\n총 {len(models)}개 모델 로드 완료: {list(models.keys())}")

In [ ]:
# 테스트 데이터 로드
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# config에서 경로 가져오기
test_data_path = config.get('local_test_path', str(PROJECT_ROOT / '1_data_preparation' / 'data' / 'test'))
print(f"테스트 데이터 경로: {test_data_path}")

test_dataset = datasets.ImageFolder(test_data_path, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# ============================================
# 📊 모든 모델 평가
# ============================================

def evaluate_model(model, data_loader, device):
    """모델 성능 평가"""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Evaluating"):
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='binary', pos_label=0),
        'recall': recall_score(all_labels, all_preds, average='binary', pos_label=0),
        'f1_score': f1_score(all_labels, all_preds, average='binary', pos_label=0),
        'confusion_matrix': confusion_matrix(all_labels, all_preds).tolist()
    }

# 각 기법별 평가 실행
all_results = {}

print("=" * 60)
print("  📊 Fine-tuning 기법별 평가")
print("=" * 60)

for method, model in models.items():
    print(f"\n🔹 {method.upper()} 모델 평가 중...")
    results = evaluate_model(model, test_loader, device)
    all_results[method] = results
    print(f"   Accuracy: {results['accuracy']*100:.1f}%")

print("\n✅ 모든 모델 평가 완료!")

In [ ]:
# ============================================
# 📊 결과 비교 테이블
# ============================================

print("\n" + "=" * 70)
print("  📊 Fine-tuning 기법별 성능 비교 (한국인 테스트 데이터)")
print("=" * 70)
print(f"{'기법':<12} {'Accuracy':>12} {'Precision':>12} {'Recall':>12} {'F1 Score':>12}")
print("-" * 70)

for method, results in all_results.items():
    print(f"{method.upper():<12} {results['accuracy']*100:>11.1f}% {results['precision']*100:>11.1f}% {results['recall']*100:>11.1f}% {results['f1_score']*100:>11.1f}%")

print("=" * 70)

# 최고 성능 기법 찾기
best_method = max(all_results.keys(), key=lambda m: all_results[m]['accuracy'])
best_acc = all_results[best_method]['accuracy']

print(f"\n🏆 최고 성능: {best_method.upper()} ({best_acc*100:.1f}%)")
print("\n해석:")
print("  - Full: 전체 파라미터 학습으로 가장 높은 성능 (과적합 주의)")
print("  - Freeze: 빠르고 안정적이지만 성능 제한")
print("  - LoRA: 효율적이면서 좋은 성능 (LLM에서 인기)")

In [ ]:
# ============================================
# 💾 결과 저장
# ============================================

# 모든 기법 결과 저장
for method, results in all_results.items():
    result_file = f'{method}_results.json'
    with open(result_file, 'w') as f:
        json.dump({'model_type': f'after_{method}', **results}, f, indent=2)
    print(f"✅ {method.upper()} 결과 저장: {result_file}")

# 통합 결과 저장 (5_comparison에서 사용)
all_after_results = {
    'methods': list(all_results.keys()),
    'results': all_results,
    'best_method': best_method,
    'best_accuracy': best_acc
}

with open('after_results.json', 'w') as f:
    json.dump(all_after_results, f, indent=2)
print(f"\n✅ 통합 결과 저장: after_results.json")

# config 업데이트
config['after_results'] = all_results
config['best_method'] = best_method
config['after_accuracy'] = best_acc

config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config 업데이트 완료")

<cell_type>markdown</cell_type>## 4.1 Model Registry 등록

### Model Registry란?
- ML 모델의 **버전 관리** 시스템
- 모델 승인 워크플로우 (Pending → Approved → Deployed)
- 모델 메타데이터 및 성능 지표 추적

### 등록 조건
- 정확도 **85% 이상**인 모델만 등록
- 최고 성능 기법의 모델을 등록

```
모델 학습 완료
      ↓
[정확도 >= 85%?]
    Yes → Model Registry 등록 (PendingManualApproval)
    No  → 등록 건너뜀
      ↓
관리자 승인 후 배포 가능
```

In [ ]:
from sagemaker.pytorch import PyTorchModel

MODEL_PACKAGE_GROUP = "deepfake-detection-kodf"
ACCURACY_THRESHOLD = 0.85

# Model Package Group 생성 (처음 한 번만)
sm_client = boto3.client('sagemaker')
try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription="한국인 딥페이크 탐지 모델"
    )
    print(f"Model Package Group 생성: {MODEL_PACKAGE_GROUP}")
except sm_client.exceptions.ClientError as e:
    if "already exists" in str(e):
        print(f"Model Package Group 이미 존재: {MODEL_PACKAGE_GROUP}")
    else:
        raise e

In [ ]:
# ============================================
# 📦 최고 성능 모델 Model Registry 등록
# ============================================
from sagemaker.pytorch import PyTorchModel

ACCURACY_THRESHOLD = 0.85

# 최고 성능 기법의 모델 경로
best_model_data = training_results[best_method]['model_data']
best_accuracy = all_results[best_method]['accuracy']

print(f"최고 성능 기법: {best_method.upper()}")
print(f"정확도: {best_accuracy*100:.1f}%")
print(f"모델 경로: {best_model_data}")

if best_accuracy >= ACCURACY_THRESHOLD:
    print(f"\n✅ 정확도 {best_accuracy*100:.1f}% >= {ACCURACY_THRESHOLD*100}% 기준 충족!")
    print("Model Registry에 등록합니다...")
    
    # inference.py 경로
    inference_dir = str(PROJECT_ROOT / '6_demo')
    
    # PyTorch 모델 정의
    pytorch_model = PyTorchModel(
        model_data=best_model_data,
        role=config['role'],
        framework_version='2.0.0',
        py_version='py310',
        entry_point='inference.py',
        source_dir=inference_dir
    )
    
    # Model Registry 등록
    model_package = pytorch_model.register(
        model_package_group_name=MODEL_PACKAGE_GROUP,
        inference_instances=['ml.g4dn.xlarge', 'ml.m5.large'],
        transform_instances=['ml.m5.large'],
        content_types=['application/x-image', 'application/json'],
        response_types=['application/json'],
        approval_status='PendingManualApproval',
        description=f"{best_method.upper()} Fine-tuned (Accuracy: {best_accuracy*100:.1f}%)"
    )
    
    print(f"\n✅ Model Registry 등록 완료!")
    print(f"   Model Package ARN: {model_package.model_package_arn}")
    print(f"   상태: PendingManualApproval (관리자 승인 필요)")
    
    config['model_package_arn'] = model_package.model_package_arn
    config['registered_method'] = best_method
    
else:
    print(f"\n❌ 정확도 {best_accuracy*100:.1f}% < {ACCURACY_THRESHOLD*100}% 기준 미달")
    print("Model Registry 등록 건너뜀")
    print("팁: 에포크 수를 늘리거나 학습 데이터를 추가해보세요.")

# config 저장
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

<cell_type>markdown</cell_type>## 완료!

Fine-tuned 모델 평가가 완료되었습니다.

### 결과 요약
- 각 Fine-tuning 기법별 성능을 테스트 데이터로 평가
- 최고 성능 모델을 Model Registry에 등록 (85%+ 시)

### 다음 단계에서 확인할 것
- Before vs After 성능 비교
- 기법별 효율성 분석 (파라미터 수 대비 성능)

**➡️ 다음 단계: `5_comparison/compare_results.ipynb`**

Before 모델과 각 Fine-tuning 기법의 성능을 종합 비교합니다!